# Experimento: índice de marginación y accesibilidad a servicios de salud

> **Experimental** (rama `exp/indice-marginacion`). La fuente CONAPO-MARG-001 está en estado `proposed` y no
> entra al pipeline ni a la tabla final.

Preguntas:

1. ¿Cómo se relaciona la marginación de una localidad con su acceso a servicios de salud?
2. ¿Una localidad con acceso comprometido es **más vulnerable** si además tiene marginación alta? ¿Cuántas son y dónde están?
3. ¿Qué cambiaría en la priorización si se tomara en cuenta la marginación?

Fuente: CONAPO, *Índice de marginación por localidad 2020* (método DP2), calculado con el Censo 2020, igual que ITER.
Ocho indicadores: analfabetismo, población sin educación básica, viviendas sin drenaje ni excusado, sin
electricidad, sin agua entubada, con piso de tierra, con hacinamiento y sin refrigerador.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from health_data import analisis, marginacion  # noqa: E402
from health_data.config import ARCHIVO_TABLA_FINAL, cargar_configuracion  # noqa: E402
from health_data.processing import conapo_marg_001  # noqa: E402

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.1f}".format)
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"

UMBRALES = cargar_configuracion().umbrales
U1 = analisis.umbral_critico("primer", UMBRALES)
U2 = analisis.umbral_critico("segundo", UMBRALES)
COLORES_GRADO = {
    "MUY BAJO": "#2a78d6",
    "BAJO": "#7fb0e8",
    "MEDIO": "#B8B8B8",
    "ALTO": "#eb6834",
    "MUY ALTO": "#B00020",
}
print(f"Umbral crítico: 1er nivel > {U1:.0f} min · 2do nivel > {U2:.0f} min")

## 1. Datos y cobertura del índice

Si faltan las tablas procesadas de CONAPO se generan desde `data/raw/conapo_iml2020_localidades.zip`.

CONAPO solo calcula el índice de las localidades con **3 viviendas habitadas o más**. Las de 1 y 2 viviendas
vienen agregadas por municipio; a esas localidades se les asigna el agregado de su municipio
(`marginacion_origen = agregado_municipal`). Ese valor describe al conjunto de ranchos del municipio, no a
cada uno, así que se reporta aparte.

In [ ]:
if not (conapo_marg_001.ARCHIVO_LOCALIDADES.exists() and conapo_marg_001.ARCHIVO_AGREGADOS.exists()):
    conapo_marg_001.main()

tabla = marginacion.unir_marginacion(
    pd.read_parquet(ARCHIVO_TABLA_FINAL),
    pd.read_parquet(conapo_marg_001.ARCHIVO_LOCALIDADES),
    pd.read_parquet(conapo_marg_001.ARCHIVO_AGREGADOS),
)
tabla = analisis.preparar_tabla(tabla)
tabla["comprometido"] = analisis.acceso_comprometido(tabla, umbrales=UMBRALES)

cobertura = tabla.groupby("marginacion_origen", dropna=False).agg(
    localidades=("population", "size"),
    poblacion=("population", "sum"),
    localidades_comprometidas=("comprometido", "sum"),
    poblacion_comprometida=("population", lambda s: s[tabla.loc[s.index, "comprometido"]].sum()),
)
for col in ["localidades", "poblacion", "localidades_comprometidas", "poblacion_comprometida"]:
    cobertura[f"pct_{col}"] = 100 * cobertura[col] / cobertura[col].sum()
cobertura

El índice propio cubre pocas localidades pero casi toda la población, **también entre quienes tienen acceso
comprometido**: las localidades de 1 y 2 viviendas son muchas pero tienen muy poca gente.

**Cómo leer el índice.** En el método DP2 un índice **mayor** indica **menor** marginación. Para que sea más
intuitivo, `marginacion_norm = 1 − IMN_2020` va de 0 a 1 y un valor mayor es más marginación. El grado
(`MUY BAJO` a `MUY ALTO`) es la forma oficial de reportarlo.

In [ ]:
propias = tabla[tabla["marginacion_origen"].eq(marginacion.ORIGEN_LOCALIDAD)]
distribucion = propias.groupby("grado_marginacion", observed=False).agg(
    localidades=("population", "size"), poblacion=("population", "sum")
)
distribucion["pct_poblacion"] = 100 * distribucion["poblacion"] / distribucion["poblacion"].sum()
distribucion

## 2. ¿La marginación se relaciona con el acceso?

Acceso comprometido por grado de marginación, solo con localidades que tienen índice propio (valores medidos
en la propia localidad). `tiempo_segundo_ponderado` es el tiempo promedio al hospital ponderado por población.

In [ ]:
por_grado = marginacion.acceso_por_grado(tabla, umbrales=UMBRALES)
por_grado

In [ ]:
fig = px.bar(
    por_grado.reset_index(),
    x="grado_marginacion",
    y="pct_poblacion_comprometida",
    color="grado_marginacion",
    color_discrete_map=COLORES_GRADO,
    text_auto=".1f",
    labels={"grado_marginacion": "Grado de marginación", "pct_poblacion_comprometida": "% de la población con acceso comprometido"},
    title="Población con acceso comprometido según el grado de marginación de su localidad",
)
fig.update_layout(showlegend=False, height=450)
fig.show()

Con el agregado municipal para las localidades de 1 y 2 viviendas (todas las localidades):

In [ ]:
marginacion.acceso_por_grado(tabla, solo_indice_propio=False, umbrales=UMBRALES)

### Sobrerrepresentación

Igual que con los grupos vulnerables del notebook 2.0: ¿qué parte de la población con acceso comprometido vive
en localidades de marginación alta o muy alta, frente a la del estado?

In [ ]:
alta = tabla["grado_marginacion"].isin(marginacion.GRADOS_ALTOS)
filas = []
for nombre, base in [("Índice propio", tabla["marginacion_origen"].eq(marginacion.ORIGEN_LOCALIDAD)),
                     ("Con agregado municipal", tabla["marginacion_origen"].notna())]:
    t = tabla[base]
    a = alta[base]
    filas.append({
        "base": nombre,
        "pct_alta_en_el_estado": 100 * t.loc[a, "population"].sum() / t["population"].sum(),
        "pct_alta_entre_comprometidos": 100 * t.loc[a & t["comprometido"], "population"].sum()
        / t.loc[t["comprometido"], "population"].sum(),
    })
sobre = pd.DataFrame(filas).set_index("base")
sobre["veces"] = sobre["pct_alta_entre_comprometidos"] / sobre["pct_alta_en_el_estado"]
sobre

## 3. ¿La marginación aporta información nueva?

Correlación de Spearman (por rangos, sin ponderar por población) entre la marginación y los tiempos y las
variables de vulnerabilidad que ya tiene la tabla final, en localidades con índice propio. Una correlación
baja indica que la marginación mide algo distinto y vale la pena agregarla; una muy alta, que es redundante.

In [ ]:
variables = {
    "primary_time_min": "Tiempo a centro de salud",
    "secondary_time_min": "Tiempo a hospital",
    "pct_viviendas_sin_auto_ni_moto": "% viviendas sin auto ni moto",
    "pct_lengua_indigena": "% hablante de lengua indígena",
    "pct_sin_afiliacion": "% sin afiliación",
    "pct_60_y_mas": "% 60 años y más",
    "pct_con_discapacidad": "% con discapacidad",
}
datos = propias[["marginacion_norm", *variables]].apply(pd.to_numeric, errors="coerce")
correlacion = (
    datos.corr(method="spearman")["marginacion_norm"].drop("marginacion_norm").rename(index=variables).to_frame("spearman")
)
correlacion["localidades_con_dato"] = [int(datos[["marginacion_norm", c]].dropna().shape[0]) for c in variables]
correlacion.style.format({"spearman": "{:.2f}"})

## 4. Prioridad: acceso comprometido × marginación alta

Cada localidad se clasifica en cuatro grupos. Marginación alta = grado `ALTO` o `MUY ALTO`.

1. **Acceso comprometido y marginación alta** (doble carga): lejos del servicio y con menos recursos para llegar o resolverlo de otra forma.
2. Acceso comprometido, marginación media o baja.
3. Marginación alta, acceso adecuado: el problema principal no es la distancia.
4. Sin alerta.

In [ ]:
tabla["prioridad"] = marginacion.clasificar_prioridad(tabla, umbrales=UMBRALES)
marginacion.resumen_prioridad(tabla, umbrales=UMBRALES)

In [ ]:
fig = px.scatter(
    propias.assign(prioridad=tabla.loc[propias.index, "prioridad"]),
    x="secondary_time_min",
    y="marginacion_norm",
    size="population",
    size_max=40,
    color="grado_marginacion",
    color_discrete_map=COLORES_GRADO,
    category_orders={"grado_marginacion": marginacion.GRADOS},
    hover_name="locality_name",
    hover_data={"municipality_name": True, "population": ":,", "primary_time_min": ":.0f", "secondary_time_min": ":.0f"},
    log_x=True,
    labels={"secondary_time_min": "Minutos al hospital (escala log)", "marginacion_norm": "Marginación (1 − IMN)",
            "grado_marginacion": "Grado"},
    title="Tiempo al hospital y marginación (localidades con índice propio; tamaño = población)",
)
fig.add_vline(x=U2, line_dash="dash", line_color="#555", annotation_text=f"{U2:.0f} min")
fig.update_layout(height=550)
fig.show()

## 5. Localidades con doble carga

Primero las de índice propio (confirmadas); después las que entran por el agregado de su municipio
(probables: el grado es del conjunto de ranchos del municipio).

In [ ]:
doble = marginacion.localidades_doble_carga(tabla, umbrales=UMBRALES)
confirmadas = doble[doble["marginacion_origen"].eq(marginacion.ORIGEN_LOCALIDAD)]
print(f"Doble carga: {len(doble):,} localidades, {doble['population'].sum():,.0f} personas "
      f"({len(confirmadas)} con índice propio, {confirmadas['population'].sum():,.0f} personas)")
confirmadas.head(25)

In [ ]:
(
    doble.groupby(["municipality_name", "marginacion_origen"]).agg(localidades=("population", "size"), poblacion=("population", "sum"))
    .unstack("marginacion_origen", fill_value=0)
    .sort_values(("poblacion", marginacion.ORIGEN_LOCALIDAD), ascending=False)
)

## 6. ¿Qué cambia en la priorización?

Las localidades con acceso comprometido más pobladas, que hoy encabezan la lista del notebook 2.0, y su grado
de marginación:

In [ ]:
(
    tabla[tabla["comprometido"]]
    .sort_values("population", ascending=False)
    .head(15)[["locality_name", "municipality_name", "population", "primary_time_min", "secondary_time_min",
               "grado_marginacion", "marginacion_origen"]]
    .reset_index(drop=True)
)

Y las localidades con marginación alta pero acceso adecuado (grupo 3), por municipio. Aquí la marginación
señala vulnerabilidad que la distancia no capta.

In [ ]:
(
    tabla[tabla["prioridad"].eq(marginacion.PRIORIDADES[2])]
    .groupby("municipality_name")
    .agg(localidades=("population", "size"), poblacion=("population", "sum"),
         lengua_indigena=("poblacion_lengua_indigena", "sum"))
    .sort_values("poblacion", ascending=False)
    .head(10)
)

## 7. Conclusiones (cifras al 14/11/2026)

1. **Sí hay relación, y es fuerte cuando se pondera por población.** Entre localidades con índice propio, la
   población con acceso comprometido pasa de 0.5 % con marginación muy baja a 4.9 % con media, 13.1 % con alta y
   23.5 % con muy alta. El tiempo promedio al hospital va de 8 a 79 min.
2. **La marginación alta está sobrerrepresentada entre quienes tienen acceso comprometido:** el 7.6 % de esa
   población vive en localidades de marginación alta o muy alta, contra el 0.3 % del estado (unas 24 veces más).
3. **Aporta información nueva.** Por localidad, su correlación con el tiempo al hospital es baja (Spearman 0.13) y
   también con las variables de vulnerabilidad de la tabla, salvo la falta de vehículo (0.48). No es redundante.
4. **Doble carga:** 60 localidades confirmadas (1,498 personas) y 163 probables por el agregado municipal
   (en total 223 y 1,933 personas). Se concentran en Yécora, Álamos, Quiriego, Sahuaripa y Rosario.
5. **Cambia el orden, no el tamaño del problema.** Las localidades comprometidas más pobladas (Puerto Libertad,
   Yécora, Suaqui Grande, Sásabe) tienen marginación muy baja: siguen siendo prioridad por población. La
   marginación sirve para ordenar dentro del grupo comprometido y para señalar las rancherías de la sierra
   (Yécora, Álamos, Quiriego) donde coinciden distancia y carencias.
6. **Marginación alta con acceso adecuado** (grupo 3, ~8,500 personas): sobre todo comunidades yaquis de Guaymas
   y localidades de Álamos. No es un problema de distancia, sino de otro tipo de vulnerabilidad.

### Limitaciones

- 5,447 localidades (1 y 2 viviendas) solo tienen el agregado municipal; su grado es una aproximación.
- El índice es de 2020 y las rutas de 2026; la marginación cambia lento, pero no es simultáneo.
- La clasificación usa umbrales fijos (120 min al hospital, grado alto o muy alto); con otros umbrales cambian
  los conteos.
- Es una asociación, no causalidad: la marginación y la lejanía comparten causas (sierra, dispersión).

### Siguientes pasos posibles

- Pasar CONAPO-MARG-001 a `approved` tras la revisión del equipo e integrar `grado_marginacion` a la tabla final.
- Agregar al tablero un filtro o capa por grado de marginación.
- Probar un índice compuesto (tiempo × marginación) solo si el equipo acuerda los pesos; la clasificación en
  cuatro grupos es más fácil de explicar.